# 15. European benchmark

Place Portugal's subsector composition in the distribution of European reporters, which is the one question a single-country study cannot answer.

**Reads**

- `data/processed/european_subsector_panel_1995_2025.csv`
- `outputs/tables/european_benchmark_summary.csv`
- `outputs/tables/european_benchmark_position.csv`

**Writes**

- Nothing. All three artefacts are persisted by the pipeline.

**Method reference:** `METHODOLOGY.md` section 17

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

%matplotlib inline

from portugal_fiscal_balance.analysis import figures

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. Why leave Portugal at all

Every other notebook describes one country. That establishes how Portugal behaves and
cannot establish whether it is unusual, because there is no comparison set. ESA 2010
requires the same subsector breakdown from every reporter, so the comparison exists.

Three construction choices decide whether the comparison is fair.

**The non-Social-Security aggregate includes state government.**

$$B^{nonSSF}_{c,t} = B^{S.1311}_{c,t} + B^{S.1312}_{c,t} + B^{S.1313}_{c,t}.$$

Portugal has no S.1312 tier; Germany, Spain, Austria, Belgium and Switzerland do.
Omitting it would leave their identity open and understate their non-Social-Security
deficits. A missing tier contributes zero because it does not exist, not because a
value is unknown.

**Ratios use national currency.** Eurostat publishes shares of GDP to one decimal,
which is an unusable denominator: a balance printed as -0.2 could sit anywhere in a
band wide enough to move the offset ratio by a quarter of its value.

**Coverage is made comparable.** A country-year needs all four required sectors, and a
reporter needs at least fifteen complete years before its frequencies are compared with
a reporter covering thirty.

In [ ]:
panel = pd.read_csv(PROCESSED / 'european_subsector_panel_1995_2025.csv')
summary = pd.read_csv(TABLES / 'european_benchmark_summary.csv')
print('country-years:', len(panel), 'of which complete:', int(panel['complete'].sum()))
print('reporters in the summary:', len(summary))
print('offset ratio defined in:', int(panel['offset_ratio'].notna().sum()), 'country-years')
print('max |identity residual|, national currency:', round(float(panel['closure_error_mio_nac'].abs().max()), 2))

## 2. Does the external source agree with our own panel?

Eurostat compiles the Portuguese figures independently of the PORDATA bridge this
repository uses. Agreement is therefore a check on the extraction, not a tautology.

In [ ]:
domestic = pd.read_csv(PROCESSED / 'fiscal_balances_1977_2025.csv')
check = (
    panel.loc[panel['country'].eq('PT'), ['year', 'general_government_mio_nac', 'social_security_mio_nac']]
    .merge(
        domestic[['year', 'general_government_balance_m_eur', 'social_security_balance_m_eur']],
        on='year',
    )
)
check['gg_gap'] = check['general_government_mio_nac'] - check['general_government_balance_m_eur']
check['ssf_gap'] = check['social_security_mio_nac'] - check['social_security_balance_m_eur']
print('years compared:', len(check))
print('max |General Government gap| (M EUR):', round(float(check['gg_gap'].abs().max()), 2))
print('max |Social Security gap| (M EUR):', round(float(check['ssf_gap'].abs().max()), 2))
display(check.tail(4).round(2))

## 3. Sign frequencies across reporters

Ordered by the frequency of a Social Security surplus, because the position in the
ranking is the quantity of interest.

In [ ]:
display(
    summary.sort_values('share_ssf_positive', ascending=False)[
        [
            'country',
            'n_years',
            'share_central_negative',
            'share_ssf_positive',
            'mean_ssf_pct_gdp',
            'median_offset_ratio',
        ]
    ].round(3)
)

In [ ]:
figure = figures.european_benchmark(summary)

In [ ]:
figure = figures.european_offset_distribution(panel)

## 4. Where Portugal sits

The answer is not uniform, which is the substantive result. A persistently
deficit-running central tier is common in Europe. A Social Security surplus of
Portugal's frequency and size is not.

In [ ]:
position = pd.read_csv(TABLES / 'european_benchmark_position.csv')
display(position.round(3))
central_always = summary.loc[summary['share_central_negative'].ge(1.0), 'country'].tolist()
print('reporters with a Central Government deficit in every year:', len(central_always))
print(' ', central_always)

## 5. The composition of a surplus year

This is the paper's headline composition restated as a cross-country question: when a
country records an aggregate surplus, is its non-Social-Security balance in deficit?

In [ ]:
with_surplus = summary.loc[summary['n_aggregate_positive'].gt(0)].copy()
with_surplus['share_offsetting'] = (
    with_surplus['n_aggregate_positive_with_negative_non_ssf'] / with_surplus['n_aggregate_positive']
)
display(
    with_surplus.sort_values('share_offsetting', ascending=False)[
        ['country', 'n_aggregate_positive', 'n_aggregate_positive_with_negative_non_ssf', 'share_offsetting']
    ].round(3)
)
total = int(with_surplus['n_aggregate_positive'].sum())
offsetting = int(with_surplus['n_aggregate_positive_with_negative_non_ssf'].sum())
print(f'surplus country-years: {offsetting} of {total} have a negative non-SSF balance ({100 * offsetting / total:.0f}%)')
print('reporters showing it in every surplus year:', int((with_surplus['share_offsetting'] >= 1.0).sum()))

## Interpretation limits

1. **This is a distribution, not a test.** Locating Portugal in a spread of accounting
   compositions says how common that composition is. It says nothing about why any
   country's composition takes the form it does.
2. **Nothing is held constant.** Reporters differ in whether they operate a state tier,
   in how contributory schemes are assigned between tiers, in pension-system maturity
   and in how transfers are routed between subsectors.
3. **Portugal's surplus years are few.** A share computed over four observations is
   reported with its count beside it and should not be read as a rate.
4. **Eurostat vintages differ from the domestic sources.** The agreement checked in
   section 2 is to rounding, not to the euro.
5. Excluding reporters with fewer than fifteen complete years is a **stated choice**,
   not a property of the data; the excluded reporters remain in the persisted panel.

---

[Previous: 14. Descriptive macroeconomic co-movement](14_macroeconomic_comovement.ipynb) | [Next: 16. Build the final report](16_build_report.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```